In [ ]:
!pip -q install langchain langchain-community chromadb beautifulsoup4
!pip -q install openai openai-whisper
!pip -q install gradio groq

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/vet_doctor_agent

In [ ]:
import os
print(os.listdir('.'))

In [ ]:
from openai import OpenAI
from groq import Groq

from modules.rag_store import build_vectorstore, get_retriever
from modules.video_utils import extract_frames_by_seconds
from modules.audio_utils import transcribe_video_audio
from modules.summarizer import build_case_summary
from modules.triage import safe_triage

openai_client = OpenAI()
groq_client = Groq()

urls = [
  "https://www.merckvetmanual.com/",
  "https://www.avma.org/dog-health/"
  "https://bondvet.com/blog/why-is-my-dog-throwing-up?sem_account_id=6523229862&sem_campaign_id=22311144632&sem_ad_group_id=&sem_device_type=c&sem_ad_id=&sem_network=x&utm_source=google&utm_medium=cpc&utm_term=&utm_campaign=OVR%20%7C%20PMAX%20%7C%20Queens&gad_source=1&gad_campaignid=22418116220&gbraid=0AAAAACwuCYzAXKRz5rYLMhpV04mrV3bL-&gclid=EAIaIQobChMI5Pbf48iTkwMVZUT_AR3WhTLWEAAYAyAAEgLwZ_D_BwE", # vomit
  "https://www.petmd.com/dog/symptoms/dog-diarrhea", # diarrhea
  "https://bondvet.com/blog/kennel-cough-in-dogs?sem_account_id=6523229862&sem_campaign_id=22311144632&sem_ad_group_id=&sem_device_type=c&sem_ad_id=&sem_network=x&utm_source=google&utm_medium=cpc&utm_term=&utm_campaign=OVR%20%7C%20PMAX%20%7C%20Queens&gad_source=1&gad_campaignid=22418116220&gbraid=0AAAAACwuCYzAXKRz5rYLMhpV04mrV3bL-&gclid=EAIaIQobChMI6tmQmsmTkwMVdTQIBR2Q7DbSEAAYASAAEgKQdfD_BwE", # cough
  "https://www.vet.cornell.edu/departments-centers-and-institutes/riney-canine-health-center/canine-health-topics/recognizing-and-responding-canine-respiratory-distress", # difficult breathing
  "https://www.petmd.com/dog/symptoms/why-is-my-dog-limping", # limping
  "https://www.petmd.com/dog/symptoms/lethargy-in-dogs", # lethargy
  "https://www.petmd.com/blogs/thedailyvet/dr-coates/2016/april/fly-biting-it-seizure-or-digestive-disorder-33881", # seizure
  "https://www.petmd.com/ferret/conditions/digestive/c_ft_anorexia",
  "https://www.petmd.com/dog/general-health/dog-eye-discharge",
  "https://www.petmd.com/dog/general-health/signs-of-dog-ear-infection",
  "https://www.petmd.com/pet-medication/rimadyl-carprofen",
  "https://www.petmd.com/dog/conditions/systemic/heatstroke-dogs",
  "https://www.petmd.com/dog/general-health/do-dogs-sweat",
  "https://www.petmd.com/dog/care/pedialyte-dogs-it-safe",
  "https://www.petmd.com/dog/symptoms/is-my-dog-dehydrated",
  "https://www.petmd.com/dog/general-health/pain-meds-for-dogs",
  "https://www.petmd.com/dog/general-health/back-pain-in-dogs",
  "https://www.petmd.com/dog/general-health/signs-a-dog-is-dying-of-cancer",

]
vs, chunks = build_vectorstore(urls, persist_dir="vet_db")
retriever = get_retriever(vs, k=4)
print("chunks:", len(chunks))

In [ ]:
import tempfile, shutil
from modules.video_utils import extract_frames_by_seconds
from modules.audio_utils import transcribe_video_audio
from modules.summarizer import build_case_summary
from modules.triage import safe_triage

def normalize_video_input(video_file):
    if video_file is None:
        return None
    if isinstance(video_file, str):
        return video_file
    if isinstance(video_file, dict):
        return video_file.get("path") or video_file.get("name")
    return str(video_file)

def run_pipeline(
    video_file=None,
    symptom_text="",
    every_sec=2.0,
    max_frames=10,
    whisper_model="base",
    max_repairs=1
):
    symptom_text = (symptom_text or "").strip()
    video_path = normalize_video_input(video_file)

    # both empty -> reject
    if not video_path and not symptom_text:
        triage_obj = {
            "triage": "unknown",
            "final_summary": "Please upload a video or enter symptom notes.",
            "key_risks": [],
            "what_to_monitor": [],
            "when_to_seek_vet": [],
            "vet_visit_checklist": [],
            "citations": [],
            "disclaimer": "This tool is for informational use only and is not a diagnosis."
        }
        debug_out = {
            "audio_transcript_preview": "",
            "query": None,
            "sources": None,
            "evidence_preview": "",
            "raw_model_output": "",
            "case_summary": "",
            "user_symptom_text": symptom_text
        }
        return triage_obj, debug_out

    frames = []
    audio_text = ""
    frames_dir = tempfile.mkdtemp(prefix="frames_")

    try:
        if video_path:
            frames = extract_frames_by_seconds(
                video_path,
                out_dir=frames_dir,
                every_sec=every_sec,
                max_frames=max_frames
            )
            audio_text = transcribe_video_audio(
                video_path,
                whisper_model=whisper_model
            )

        # combine transcript + user text
        combined_text_parts = []
        if audio_text.strip():
            combined_text_parts.append(f"Audio transcript:\n{audio_text.strip()}")
        if symptom_text:
            combined_text_parts.append(f"Owner notes:\n{symptom_text}")

        combined_text = "\n\n".join(combined_text_parts).strip()

        case_summary = build_case_summary(
            frames,
            combined_text,
            client=openai_client,
            model="gpt-4o-mini"
        )

        triage_obj, debug = safe_triage(
            case_summary,
            combined_text,
            retriever,
            groq_client,
            max_repairs=max_repairs,
            verbose=False
        )

        debug_out = {
            "audio_transcript_preview": audio_text[:1200],
            "user_symptom_text": symptom_text,
            "combined_text_preview": combined_text[:1500],
            "query": debug.get("query"),
            "sources": debug.get("sources"),
            "evidence_preview": (debug.get("evidence") or "")[:2500],
            "raw_model_output": (debug.get("raw") or "")[:2500],
            "case_summary": case_summary
        }
        return triage_obj, debug_out

    finally:
        shutil.rmtree(frames_dir, ignore_errors=True)

In [ ]:
import json
import gradio as gr

def _to_md_list(items):
    if not items:
        return "_(none)_"
    return "\n".join([f"- {x}" for x in items])

def run_and_render(video_file, symptom_text, every_sec, max_frames, whisper_model, max_repairs):
    triage_obj, debug_obj = run_pipeline(
        video_file=video_file,
        symptom_text=symptom_text,
        every_sec=every_sec,
        max_frames=max_frames,
        whisper_model=whisper_model,
        max_repairs=max_repairs
    )

    triage_level = (triage_obj.get("triage") or "").lower()
    badge = {
        "red": "🔴 **RED (Urgent)**",
        "yellow": "🟡 **YELLOW (Soon)**",
        "green": "🟢 **GREEN (Monitor)**",
    }.get(triage_level, f"⚪ **{triage_obj.get('triage','unknown')}**")

    summary = triage_obj.get("final_summary", "")

    risks_md = _to_md_list(triage_obj.get("key_risks", []))
    monitor_md = _to_md_list(triage_obj.get("what_to_monitor", []))
    when_md = _to_md_list(triage_obj.get("when_to_seek_vet", []))
    checklist_md = _to_md_list(triage_obj.get("vet_visit_checklist", []))

    citations = triage_obj.get("citations", [])
    citations_md = "\n".join([f"- {c}" for c in citations]) if citations else "_(none)_"
    disclaimer = triage_obj.get("disclaimer", "")

    transcript_preview = debug_obj.get("audio_transcript_preview", "")
    evidence_preview = debug_obj.get("evidence_preview") or debug_obj.get("evidence") or ""
    triage_json = json.dumps(triage_obj, ensure_ascii=False, indent=2)
    debug_json = json.dumps(debug_obj, ensure_ascii=False, indent=2)

    return (
        badge,
        summary,
        risks_md,
        monitor_md,
        when_md,
        checklist_md,
        citations_md,
        disclaimer,
        transcript_preview,
        evidence_preview,
        triage_json,
        debug_json
    )

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🐶 Vet Triage Agent Demo")

    with gr.Row():
        with gr.Column(scale=5):
            video = gr.Video(label="Upload dog video (optional)", format="mp4")

            symptom_text = gr.Textbox(
                label="Describe symptoms or concerns (optional)",
                placeholder="e.g. My dog vomited 3 times today, seems tired, and is not eating much.",
                lines=4
            )

            with gr.Accordion("⚙️ Advanced settings", open=False):
                every_sec = gr.Slider(0.5, 5.0, value=2.0, step=0.5, label="Frame sampling interval (sec)")
                max_frames = gr.Slider(1, 30, value=10, step=1, label="Max frames")
                whisper_model = gr.Dropdown(["tiny", "base", "small", "medium"], value="base", label="Whisper model")
                max_repairs = gr.Slider(0, 3, value=1, step=1, label="Max JSON repair attempts")

            run_btn = gr.Button("🚀 Run Triage", variant="primary")

            gr.Markdown(
                "Provide a video, text notes, or both. At least one input is required."
            )

        with gr.Column(scale=7):
            badge = gr.Markdown("—")
            summary = gr.Textbox(label="Summary (non-diagnostic)", lines=3)

            with gr.Tabs():
                with gr.Tab("Risks"):
                    risks_md = gr.Markdown()
                with gr.Tab("Monitor"):
                    monitor_md = gr.Markdown()
                with gr.Tab("When to seek vet"):
                    when_md = gr.Markdown()
                with gr.Tab("Checklist"):
                    checklist_md = gr.Markdown()

            citations_md = gr.Markdown(label="Citations (sources)")
            disclaimer = gr.Textbox(label="Disclaimer", lines=2)

    with gr.Accordion("🔎 Debug", open=False):
        transcript_preview = gr.Textbox(label="Audio transcript preview", lines=4)
        evidence_preview = gr.Textbox(label="Retrieved evidence preview", lines=6)
        triage_json = gr.Code(label="Triage JSON", language="json")
        debug_json = gr.Code(label="Debug JSON", language="json")

    run_btn.click(
        fn=run_and_render,
        inputs=[video, symptom_text, every_sec, max_frames, whisper_model, max_repairs],
        outputs=[
            badge, summary,
            risks_md, monitor_md, when_md, checklist_md,
            citations_md, disclaimer,
            transcript_preview, evidence_preview,
            triage_json, debug_json
        ]
    )

demo.launch(share=True, debug=True, show_error=True)